## Set project folder and output folders for PET → Q transfer entropy analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
import seaborn as sns
import random
from matplotlib.backends.backend_pdf import PdfPages
from PyPDF2 import PdfReader
import matplotlib.dates as mdates
np.seterr(all="ignore")

{'divide': 'warn', 'over': 'warn', 'under': 'ignore', 'invalid': 'warn'}

In [2]:
# Project folder for this new PET -> Q analysis
PROJECT_DIR = Path(
    r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Transfer Entropy PET_to Q"
)

# Output folders
OUTPUT_DIR = PROJECT_DIR / "outputs"
MERGED_DIR = OUTPUT_DIR / "merged_csv"
PDF_DIR = OUTPUT_DIR / "pdf"
SUMMARY_DIR = OUTPUT_DIR / "summary_csv"
LOG_DIR = OUTPUT_DIR / "logs"

# Create folders
MERGED_DIR.mkdir(parents=True, exist_ok=True)
PDF_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR :", PROJECT_DIR)
print("MERGED_DIR  :", MERGED_DIR)
print("PDF_DIR     :", PDF_DIR)
print("SUMMARY_DIR :", SUMMARY_DIR)
print("LOG_DIR     :", LOG_DIR)

PROJECT_DIR : C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Transfer Entropy PET_to Q
MERGED_DIR  : C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Transfer Entropy PET_to Q\outputs\merged_csv
PDF_DIR     : C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Transfer Entropy PET_to Q\outputs\pdf
SUMMARY_DIR : C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Transfer Entropy PET_to Q\outputs\summary_csv
LOG_DIR     : C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Transfer Entropy PET_to Q\outputs\logs


## Define CAMELS input folders and search for daily PET-related files or columns

In [3]:
# Main CAMELS dataset folder
CAMELS_DIR = Path(
    r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\basin_timeseries_v1p2_metForcing_obsFlow\basin_dataset_public_v1p2"
)

print("CAMELS_DIR exists:", CAMELS_DIR.exists())
print("CAMELS_DIR:", CAMELS_DIR)

keywords = ["pet", "PET", "et", "ET", "evap", "EVAP", "epot", "EPOT"]
matched_files = []
column_hits = []

# Search all files
for p in CAMELS_DIR.rglob("*"):
    if not p.is_file():
        continue

    name_lower = p.name.lower()

    # file-name search
    if any(k in name_lower for k in keywords):
        matched_files.append(str(p))

    # column-name search for csv/txt files
    if p.suffix.lower() in [".csv", ".txt"]:
        try:
            df_test = pd.read_csv(p, sep=None, engine="python", nrows=5)
            cols_lower = [str(c).lower() for c in df_test.columns]
            hit_cols = [c for c in cols_lower if any(k in c for k in keywords)]
            if len(hit_cols) > 0:
                column_hits.append((str(p), hit_cols))
        except:
            pass

print("\nPET/ET-like file names found:", len(matched_files))
for f in matched_files[:50]:
    print(f)

print("\nFiles with PET/ET-like column names:", len(column_hits))
for f, cols in column_hits[:50]:
    print("\nFILE:", f)
    print("COLUMNS:", cols)

CAMELS_DIR exists: True
CAMELS_DIR: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\basin_timeseries_v1p2_metForcing_obsFlow\basin_dataset_public_v1p2

PET/ET-like file names found: 680
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\basin_timeseries_v1p2_metForcing_obsFlow\basin_dataset_public_v1p2\dataset_summary.txt
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\basin_timeseries_v1p2_metForcing_obsFlow\basin_dataset_public_v1p2\basin_metadata\basin_annual_hydrometeorology_characteristics_daymet.txt
C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\basin_timeseries_v1p2_metForcing_obsFlow\basin_dataset_public_v1p2\basin_metadata\basin_annual_hydrometeorology_characteristics_maurer.txt
C:\Users\ppaudel2\OneDrive - The University of Alabama\De

## Inspect one model output file and one parameter file to understand the file structure

In [4]:
from pathlib import Path

# Example: Daymet folder
DAYMET_DIR = Path(
    r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\basin_timeseries_v1p2_modelOutput_daymet\model_output_daymet\model_output\flow_timeseries\daymet\01"
)

# pick one basin example
file_output = DAYMET_DIR / "01013500_05_model_output.txt"
file_param  = DAYMET_DIR / "01013500_05_model_parameters.txt"

print("OUTPUT FILE EXISTS:", file_output.exists())
print("PARAM FILE EXISTS :", file_param.exists())

print("\n" + "="*80)
print("FIRST 20 LINES OF MODEL OUTPUT FILE")
print("="*80)

if file_output.exists():
    with open(file_output, "r", encoding="utf-8", errors="ignore") as f:
        for i in range(20):
            line = f.readline()
            if not line:
                break
            print(f"{i+1:02d}: {line.rstrip()}")

print("\n" + "="*80)
print("FIRST 20 LINES OF MODEL PARAMETER FILE")
print("="*80)

if file_param.exists():
    with open(file_param, "r", encoding="utf-8", errors="ignore") as f:
        for i in range(20):
            line = f.readline()
            if not line:
                break
            print(f"{i+1:02d}: {line.rstrip()}")

OUTPUT FILE EXISTS: True
PARAM FILE EXISTS : True

FIRST 20 LINES OF MODEL OUTPUT FILE
01: YR MNTH DY HR SWE PRCP RAIM TAIR PET ET MOD_RUN OBS_RUN
02: 1980 10 01 12   0.0000000   3.1000000   3.1000000   6.0800000   1.0883000   1.0828000   0.0216000   0.5510000
03: 1980 10 02 12   0.0000000   4.2400000   4.2400000  10.5300000   1.2687000   1.2624000   0.0845000   0.5607000
04: 1980 10 03 12   0.0000000   8.0200000   8.0200000  11.8350000   1.1867000   1.1807000   0.1802000   0.5586000
05: 1980 10 04 12   0.0000000  15.2700000  15.2700000   7.3800000   1.0033000   0.9983000   0.4370000   0.6712000
06: 1980 10 05 12   0.0000000   8.4800000   8.4800000   4.8000000   0.8536000   0.8493000   0.7372000   0.8216000
07: 1980 10 06 12   0.0000000   0.0000000   0.0000000   5.4100000   0.8797000   0.8753000   0.9092000   0.8671000
08: 1980 10 07 12   0.0000000   0.0000000   0.0000000   6.4050000   0.9653000   0.9567000   0.9749000   0.9115000
09: 1980 10 08 12   0.0000000   2.1000000   2.1000000  

## Compare multiple model-output versions for one basin to see whether PET changes by suffix

In [5]:
from pathlib import Path
import pandas as pd

DAYMET_DIR = Path(
    r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\basin_timeseries_v1p2_modelOutput_daymet\model_output_daymet\model_output\flow_timeseries\daymet\01"
)

basin_id = "01013500"
suffixes = ["05", "11", "27", "33", "48", "59", "66", "72"]

summary_rows = []

for s in suffixes:
    f = DAYMET_DIR / f"{basin_id}_{s}_model_output.txt"
    if not f.exists():
        continue

    df = pd.read_csv(f, sep=r"\s+", engine="python")

    row = {
        "suffix": s,
        "n_rows": len(df),
        "pet_mean": df["PET"].mean(),
        "pet_std": df["PET"].std(),
        "obs_run_mean": df["OBS_RUN"].mean(),
        "obs_run_std": df["OBS_RUN"].std(),
        "first_pet": df["PET"].iloc[0],
        "first_obs_run": df["OBS_RUN"].iloc[0],
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,suffix,n_rows,pet_mean,pet_std,obs_run_mean,obs_run_std,first_pet,first_obs_run
0,05,12510,1.951616,1.746396,1.683573,1.946997,1.0883,0.551
1,11,12510,1.956259,1.750551,1.683573,1.946997,1.0909,0.551
2,27,12510,1.950069,1.745010,1.683573,1.946997,1.0874,0.551
3,33,12510,1.950069,1.745010,1.683573,1.946997,1.0874,0.551
4,48,12510,1.950069,1.745010,1.683573,1.946997,1.0874,0.551
5,59,12510,1.960902,1.754705,1.683573,1.946997,1.0934,0.551
6,66,12510,1.951616,1.746396,1.683573,1.946997,1.0883,0.551
7,72,12510,1.950069,1.745010,1.683573,1.946997,1.0874,0.551


## Check which model-output suffixes are available across all Daymet basins

In [6]:
DAYMET_ROOT = Path(
    r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\basin_timeseries_v1p2_modelOutput_daymet\model_output_daymet\model_output\flow_timeseries\daymet"
)

rows = []

for subfolder in sorted(DAYMET_ROOT.iterdir()):
    if not subfolder.is_dir():
        continue

    for f in subfolder.glob("*_model_output.txt"):
        name = f.stem  # example: 01013500_05_model_output
        parts = name.split("_")

        if len(parts) >= 3:
            gauge_id = parts[0]
            suffix = parts[1]

            rows.append({
                "huc_folder": subfolder.name,
                "gauge_id": gauge_id,
                "suffix": suffix,
                "file_name": f.name
            })

suffix_df = pd.DataFrame(rows)

print("Total model output files found:", len(suffix_df))
print("Unique basins found:", suffix_df["gauge_id"].nunique())
print("\nSuffix counts:")
print(suffix_df["suffix"].value_counts().sort_index())

print("\nNumber of suffixes per basin:")
suffix_per_basin = suffix_df.groupby("gauge_id")["suffix"].nunique()
print(suffix_per_basin.value_counts().sort_index())

display(suffix_df.head(20))

Total model output files found: 7374
Unique basins found: 671

Suffix counts:
suffix
05    738
11    738
27    738
33    738
48    737
59    737
66    737
72    737
80    737
94    737
Name: count, dtype: int64

Number of suffixes per basin:
suffix
10    671
Name: count, dtype: int64


,huc_folder,gauge_id,suffix,file_name
0,01,01013500,05,01013500_05_model_output.txt
1,01,01013500,11,01013500_11_model_output.txt
2,01,01013500,27,01013500_27_model_output.txt
3,01,01013500,33,01013500_33_model_output.txt
4,01,01013500,48,01013500_48_model_output.txt
5,01,01013500,59,01013500_59_model_output.txt
6,01,01013500,66,01013500_66_model_output.txt
7,01,01013500,72,01013500_72_model_output.txt
8,01,01013500,80,01013500_80_model_output.txt
9,01,01013500,94,01013500_94_model_output.txt


## Build merged daily PET-Q files for all 671 basins using Daymet suffix 05

In [7]:

# Input: Daymet model-output root
DAYMET_ROOT = Path(
    r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\basin_timeseries_v1p2_modelOutput_daymet\model_output_daymet\model_output\flow_timeseries\daymet"
)

# Output: this new PET project
MERGED_DIR = PROJECT_DIR / "outputs" / "merged_csv"
LOG_DIR = PROJECT_DIR / "outputs" / "logs"

MERGED_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# choose one suffix for all basins
USE_SUFFIX = "05"

log_rows = []
count_saved = 0

for subfolder in sorted(DAYMET_ROOT.iterdir()):
    if not subfolder.is_dir():
        continue

    for f in sorted(subfolder.glob(f"*_{USE_SUFFIX}_model_output.txt")):
        gauge_id = f.stem.split("_")[0]

        try:
            df = pd.read_csv(f, sep=r"\s+", engine="python")

            # build Date column
            df["Date"] = pd.to_datetime(
                dict(year=df["YR"], month=df["MNTH"], day=df["DY"]),
                errors="coerce"
            )

            out = pd.DataFrame({
                "Date": df["Date"],
                "PET": pd.to_numeric(df["PET"], errors="coerce"),
                "Q": pd.to_numeric(df["OBS_RUN"], errors="coerce")
            })

            # keep full rows; do not drop here
            out_path = MERGED_DIR / f"{gauge_id}_PET_Q_daymet_suffix{USE_SUFFIX}.csv"
            out.to_csv(out_path, index=False)

            log_rows.append({
                "gauge_id": gauge_id,
                "suffix": USE_SUFFIX,
                "n_rows": len(out),
                "pet_nan": out["PET"].isna().sum(),
                "q_nan": out["Q"].isna().sum(),
                "status": "saved"
            })
            count_saved += 1

        except Exception as e:
            log_rows.append({
                "gauge_id": gauge_id,
                "suffix": USE_SUFFIX,
                "n_rows": np.nan,
                "pet_nan": np.nan,
                "q_nan": np.nan,
                "status": f"error: {e}"
            })

log_df = pd.DataFrame(log_rows)
log_path = LOG_DIR / f"merge_pet_q_daymet_suffix{USE_SUFFIX}_log.csv"
log_df.to_csv(log_path, index=False)

print("Merged PET-Q files saved:", count_saved)
print("Merged folder:", MERGED_DIR)
print("Log file:", log_path)

display(log_df.head())
print("\nStatus counts:")
print(log_df["status"].value_counts())

Merged PET-Q files saved: 738
Merged folder: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Transfer Entropy PET_to Q\outputs\merged_csv
Log file: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Transfer Entropy PET_to Q\outputs\logs\merge_pet_q_daymet_suffix05_log.csv


,gauge_id,suffix,n_rows,pet_nan,q_nan,status
0,01013500,05,12510,0,0,saved
1,01022500,05,12510,0,0,saved
2,01030500,05,12510,0,0,saved
3,01031500,05,12510,0,0,saved
4,01047000,05,12510,0,0,saved



Status counts:
status
saved    738
Name: count, dtype: int64


## Check actual number of saved merged files and whether any gauge IDs were repeated

In [8]:
# actual merged files in output folder
merged_files = sorted(MERGED_DIR.glob("*.csv"))
print("Actual CSV files in MERGED_DIR:", len(merged_files))

# read log file
log_path = LOG_DIR / "merge_pet_q_daymet_suffix05_log.csv"
log_df = pd.read_csv(log_path)

print("\nRows in log file:", len(log_df))
print("Unique gauge IDs in log:", log_df["gauge_id"].astype(str).str.zfill(8).nunique())

# duplicate gauge IDs in log
dup_counts = (
    log_df["gauge_id"]
    .astype(str)
    .str.zfill(8)
    .value_counts()
)

dups = dup_counts[dup_counts > 1]

print("\nNumber of gauge IDs that appear more than once in log:", len(dups))

if len(dups) > 0:
    print("\nFirst 20 duplicated gauge IDs:")
    print(dups.head(20))

Actual CSV files in MERGED_DIR: 671

Rows in log file: 738
Unique gauge IDs in log: 671

Number of gauge IDs that appear more than once in log: 27

First 20 duplicated gauge IDs:
gauge_id
01031500    4
01047000    4
01052500    4
01054200    4
01055000    4
01057000    4
01073000    4
01078000    4
01118300    4
01121000    4
01022500    4
01030500    4
01013500    4
01123000    3
01134500    3
01195100    3
04296000    3
01137500    3
01139000    3
01139800    3
Name: count, dtype: int64


## Rebuild a clean merge log from the final unique merged PET-Q files

In [9]:
merged_files = sorted(MERGED_DIR.glob("*_PET_Q_daymet_suffix05.csv"))

clean_rows = []

for f in merged_files:
    gauge_id = f.name.split("_")[0]

    try:
        df = pd.read_csv(f)

        clean_rows.append({
            "gauge_id": gauge_id,
            "suffix": "05",
            "n_rows": len(df),
            "pet_nan": pd.to_numeric(df["PET"], errors="coerce").isna().sum(),
            "q_nan": pd.to_numeric(df["Q"], errors="coerce").isna().sum(),
            "status": "saved"
        })

    except Exception as e:
        clean_rows.append({
            "gauge_id": gauge_id,
            "suffix": "05",
            "n_rows": np.nan,
            "pet_nan": np.nan,
            "q_nan": np.nan,
            "status": f"error: {e}"
        })

clean_log_df = pd.DataFrame(clean_rows).sort_values("gauge_id").reset_index(drop=True)

clean_log_path = LOG_DIR / "merge_pet_q_daymet_suffix05_log_clean.csv"
clean_log_df.to_csv(clean_log_path, index=False)

print("Clean log rows:", len(clean_log_df))
print("Unique gauge IDs:", clean_log_df["gauge_id"].nunique())
print("Clean log file:", clean_log_path)

display(clean_log_df.head())
print("\nStatus counts:")
print(clean_log_df["status"].value_counts())

Clean log rows: 671
Unique gauge IDs: 671
Clean log file: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Transfer Entropy PET_to Q\outputs\logs\merge_pet_q_daymet_suffix05_log_clean.csv


,gauge_id,suffix,n_rows,pet_nan,q_nan,status
0,01013500,05,12510,0,0,saved
1,01022500,05,12510,0,0,saved
2,01030500,05,12510,0,0,saved
3,01031500,05,12510,0,0,saved
4,01047000,05,12510,0,0,saved



Status counts:
status
saved    671
Name: count, dtype: int64


## Check one final merged PET-Q file before transfer entropy calculation

In [10]:
sample_file = MERGED_DIR / "01013500_PET_Q_daymet_suffix05.csv"

df_sample = pd.read_csv(sample_file)
df_sample["Date"] = pd.to_datetime(df_sample["Date"], errors="coerce")

print("Sample file:", sample_file.name)
print("Shape:", df_sample.shape)

print("\nColumns:")
print(df_sample.columns.tolist())

print("\nDate range:")
print(df_sample["Date"].min(), "to", df_sample["Date"].max())

print("\nNaN counts:")
print(df_sample.isna().sum())

display(df_sample.head())
display(df_sample.tail())

Sample file: 01013500_PET_Q_daymet_suffix05.csv
Shape: (12510, 3)

Columns:
['Date', 'PET', 'Q']

Date range:
1980-10-01 00:00:00 to 2014-12-31 00:00:00

NaN counts:
Date    0
PET     0
Q       0
dtype: int64


,Date,PET,Q
0,1980-10-01,1.0883,0.5510
1,1980-10-02,1.2687,0.5607
2,1980-10-03,1.1867,0.5586
3,1980-10-04,1.0033,0.6712
4,1980-10-05,0.8536,0.8216


,Date,PET,Q
12505,2014-12-27,0.1852,2.3274
12506,2014-12-28,0.1763,2.4140
12507,2014-12-29,0.0000,2.4356
12508,2014-12-30,0.0212,2.3815
12509,2014-12-31,0.0243,2.3166


## Class to compute entropy, MI, and CMI from histogram-based PDFs

In [11]:
class INFO(object):
    """
    Unified class for histogram-based information-theoretic calculations.
    Computes entropy, mutual information, and conditional mutual information.
    """

    def __init__(self, data, bins, ranges=None, weights=None, base=np.e):
        self.base = base

        if isinstance(data, pd.DataFrame):
            data = data.values

        self.data = data
        self.bins = bins
        self.ranges = ranges

        # Remove rows containing NaN
        data = data[~np.isnan(data).any(axis=1)]

        # Number of samples and dimensions
        self.N, self.D = data.shape

        # Histogram count
        pdf, self.edges = np.histogramdd(
            data,
            bins=bins,
            range=ranges,
            weights=weights,
            density=False
        )

        pdf_sum = pdf.sum()
        if pdf_sum == 0:
            raise ValueError("The histogram resulted in a zero total count.")

        # Normalize to probability
        self.pdf = pdf / pdf_sum

        assert np.isclose(self.pdf.sum(), 1.0)

    def computeEntropy(self, dims):
        base, D = self.base, self.D
        all_dims = set(range(D))

        if all_dims == set(dims):
            pdf_dim = self.pdf
        else:
            pdf_dim = self.pdf.sum(axis=tuple(all_dims - set(dims)))

        log_pdf_dim = np.ma.filled(np.log(np.ma.masked_equal(pdf_dim, 0)), 0)
        ent = -np.sum(pdf_dim * log_pdf_dim / np.log(base))
        return ent

    def computeCEntropy(self, dims, dimsc):
        hxc = self.computeEntropy(dims + dimsc)
        hc = self.computeEntropy(dimsc)
        hx_c = hxc - hc
        return hx_c

    def computeMI(self, dims1, dims2):
        h12 = self.computeEntropy(dims1 + dims2)
        h1 = self.computeEntropy(dims1)
        h2 = self.computeEntropy(dims2)
        mi = h1 + h2 - h12
        return mi

    def computeCMI(self, dims1, dims2, dimsc):
        h12c = self.computeEntropy(dims1 + dims2 + dimsc)
        h1c = self.computeEntropy(dims1 + dimsc)
        h2c = self.computeEntropy(dims2 + dimsc)
        hc = self.computeEntropy(dimsc)
        cmi = h1c + h2c - h12c - hc
        return cmi

## Define transfer entropy from PET to Q

In [12]:
def transfer_entropy(xdata, ydata, bins, base=np.e, lag=1):
    """
    Compute T(X -> Y), the transfer entropy from source xdata to target ydata.
    Here we will use:
    X = PET
    Y = Q
    """

    yt = ydata[lag:]
    x_lag = xdata[:-lag]
    y_lag1 = ydata[lag - 1:-1]

    if len(yt) == 0 or len(x_lag) == 0 or len(y_lag1) == 0:
        return 0.0

    data = np.column_stack([yt, x_lag, y_lag1])
    info = INFO(data, bins=bins, base=base)

    te_value = info.computeCMI(dims1=[1], dims2=[0], dimsc=[2])
    return te_value

## Settings for PET(t-τ) to Q(t) transfer entropy analysis

In [13]:
# SETTINGS
USE_CUSTOM_DATES = False
CUSTOM_START = "1985-01-01"
CUSTOM_END   = "2009-12-31"

NBINS = 5
LAGS = range(1, 31)

pairs = [("PET", "Q")]

out_csv = SUMMARY_DIR / "TE_AllBasins_tau1_30_PET_to_Q_daymet_suffix05_bins5.csv"
log_csv = LOG_DIR / "TE_AllBasins_PET_to_Q_daymet_suffix05_log.csv"

start_date = pd.to_datetime(CUSTOM_START)
end_date   = pd.to_datetime(CUSTOM_END)

results = []
log_rows = []

## Compute transfer entropy from PET(t-τ) to Q(t) for all 671 basins

In [14]:
for csv_path in sorted(MERGED_DIR.glob("*_PET_Q_daymet_suffix05.csv")):

    gauge_id = csv_path.name.split("_")[0]

    try:
        df = pd.read_csv(csv_path, parse_dates=["Date"]).sort_values("Date")

        # Make variables numeric
        for col in ["PET", "Q"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        # Remove invalid negative Q if any
        df.loc[df["Q"] < 0, "Q"] = np.nan

        # PET should not be negative, but keep 0 if present
        df.loc[df["PET"] < 0, "PET"] = np.nan

        # Optional date window
        if USE_CUSTOM_DATES:
            df = df[(df["Date"] >= start_date) & (df["Date"] <= end_date)].copy()

        if len(df) == 0:
            log_rows.append([gauge_id, "NO DATA", 0, "empty after date filter"])
            continue

        x_all = df["PET"].values
        y_all = df["Q"].values

        # Keep original daily alignment; do NOT remove rows before lagging
        x_use = x_all
        y_use = y_all

        for lag in LAGS:
            if len(x_use) <= lag or len(y_use) <= lag or len(x_use) < 50:
                te_val = np.nan
            else:
                te_val = transfer_entropy(
                    x_use,
                    y_use,
                    bins=NBINS,
                    base=np.e,
                    lag=lag
                )

            results.append([gauge_id, "PET", "Q", NBINS, lag, te_val])

        log_rows.append([gauge_id, "OK", len(df), ""])

        if len(log_rows) % 50 == 0:
            print("Done basins:", len(log_rows), "/ 671")

    except Exception as e:
        log_rows.append([gauge_id, "FAIL", np.nan, str(e)])

# SAVE
results_df = pd.DataFrame(
    results,
    columns=["gauge_id", "Source", "Target", "Bins", "Lag", "TE_nats"]
)
results_df.to_csv(out_csv, index=False)

log_df = pd.DataFrame(
    log_rows,
    columns=["gauge_id", "status", "n_points_window", "info"]
)
log_df.to_csv(log_csv, index=False)

print("Saved TE CSV:", out_csv.name)
print("Saved log CSV:", log_csv.name)
print(log_df["status"].value_counts())

display(results_df.head())

Done basins: 50 / 671
Done basins: 100 / 671
Done basins: 150 / 671
Done basins: 200 / 671
Done basins: 250 / 671
Done basins: 300 / 671
Done basins: 350 / 671
Done basins: 400 / 671
Done basins: 450 / 671
Done basins: 500 / 671
Done basins: 550 / 671
Done basins: 600 / 671
Done basins: 650 / 671
Saved TE CSV: TE_AllBasins_tau1_30_PET_to_Q_daymet_suffix05_bins5.csv
Saved log CSV: TE_AllBasins_PET_to_Q_daymet_suffix05_log.csv
status
OK    671
Name: count, dtype: int64


,gauge_id,Source,Target,Bins,Lag,TE_nats
0,01013500,PET,Q,5,1,0.002974
1,01013500,PET,Q,5,2,0.003436
2,01013500,PET,Q,5,3,0.003339
3,01013500,PET,Q,5,4,0.003262
4,01013500,PET,Q,5,5,0.003363


## Check the saved PET to Q transfer entropy results

In [15]:
te_pet_df = pd.read_csv(out_csv)

print("Shape:", te_pet_df.shape)
print("\nColumns:")
print(te_pet_df.columns.tolist())

print("\nUnique basins:", te_pet_df["gauge_id"].astype(str).str.zfill(8).nunique())
print("Unique lags:", te_pet_df["Lag"].nunique())

print("\nSource-Target pairs:")
print(te_pet_df[["Source", "Target"]].drop_duplicates())

print("\nMissing TE values:")
print(te_pet_df["TE_nats"].isna().sum())

display(te_pet_df.head())

Shape: (20130, 6)

Columns:
['gauge_id', 'Source', 'Target', 'Bins', 'Lag', 'TE_nats']

Unique basins: 671
Unique lags: 30

Source-Target pairs:
  Source Target
0    PET      Q

Missing TE values:
0


,gauge_id,Source,Target,Bins,Lag,TE_nats
0,1013500,PET,Q,5,1,0.002974
1,1013500,PET,Q,5,2,0.003436
2,1013500,PET,Q,5,3,0.003339
3,1013500,PET,Q,5,4,0.003262
4,1013500,PET,Q,5,5,0.003363


## Save PET to Q transfer entropy heatmaps for all basins into one PDF

In [16]:
GAUGE_INFO_PATH = Path(
    r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\basin_timeseries_v1p2_metForcing_obsFlow\basin_dataset_public_v1p2\basin_metadata\gauge_information.txt"
)

print("FILE EXISTS:", GAUGE_INFO_PATH.exists())
print("\nFIRST 15 LINES:\n")

with open(GAUGE_INFO_PATH, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(15):
        line = f.readline()
        if not line:
            break
        print(f"{i+1:02d}: {repr(line)}")

FILE EXISTS: True

FIRST 15 LINES:

01: 'HUC_02  GAGE_ID\t\t\tGAGE_NAME\t\t\t\t\tLAT\t\tLONG\t\tDRAINAGE AREA (KM^2)\n'
02: '01\t01013500\t                  Fish River near Fort Kent, Maine\t  47.23739\t -68.58264\t   2252.70\n'
03: '01\t01022500\t           Narraguagus River at Cherryfield, Maine\t  44.60797\t -67.93524\t    573.60\n'
04: '01\t01030500\t       Mattawamkeag River near Mattawamkeag, Maine\t  45.50097\t -68.30596\t   3676.17\n'
05: '01\t01031500\t      Piscataquis River near Dover-Foxcroft, Maine\t  45.17501\t -69.31470\t    769.05\n'
06: '01\t01047000\t        Carrabassett River near North Anson, Maine\t  44.86920\t -69.95510\t    909.10\n'
07: '01\t01052500\t         Diamond River near Wentworth Location, NH\t  44.87739\t -71.05749\t    383.82\n'
08: '01\t01054200\t                       Wild River at Gilead, Maine\t  44.39044\t -70.97964\t    180.98\n'
09: '01\t01055000\t                   Swift River near Roxbury, Maine\t  44.64275\t -70.58878\t    250.64\n'
10: '01\

## Load CAMELS gauge information for basin names and drainage area

In [17]:
from pathlib import Path
import pandas as pd

GAUGE_INFO_PATH = Path(
    r"C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Python\Jupyter Notebook\CAMELS Research\basin_timeseries_v1p2_metForcing_obsFlow\basin_dataset_public_v1p2\basin_metadata\gauge_information.txt"
)

gauge_df = pd.read_csv(
    GAUGE_INFO_PATH,
    sep="\t",
    skiprows=1,
    header=None,
    names=["huc_02", "gauge_id", "gauge_name", "lat", "long", "area_km2"]
)

# clean text columns
gauge_df["gauge_id"] = gauge_df["gauge_id"].astype(str).str.strip().str.zfill(8)
gauge_df["gauge_name"] = gauge_df["gauge_name"].astype(str).str.strip()

# numeric columns
gauge_df["lat"] = pd.to_numeric(gauge_df["lat"], errors="coerce")
gauge_df["long"] = pd.to_numeric(gauge_df["long"], errors="coerce")
gauge_df["area_km2"] = pd.to_numeric(gauge_df["area_km2"], errors="coerce")

print("Shape:", gauge_df.shape)
print("\nColumns:")
print(gauge_df.columns.tolist())

display(gauge_df.head())

Shape: (671, 6)

Columns:
['huc_02', 'gauge_id', 'gauge_name', 'lat', 'long', 'area_km2']


,huc_02,gauge_id,gauge_name,lat,long,area_km2
0,1,01013500,"Fish River near Fort Kent, Maine",47.23739,-68.58264,2252.70
1,1,01022500,"Narraguagus River at Cherryfield, Maine",44.60797,-67.93524,573.60
2,1,01030500,"Mattawamkeag River near Mattawamkeag, Maine",45.50097,-68.30596,3676.17
3,1,01031500,"Piscataquis River near Dover-Foxcroft, Maine",45.17501,-69.31470,769.05
4,1,01047000,"Carrabassett River near North Anson, Maine",44.86920,-69.95510,909.10


In [18]:
import gc
# Read TE results
te_df = pd.read_csv(out_csv)
te_df["gauge_id"] = te_df["gauge_id"].astype(str).str.zfill(8)

# Basin name map
gauge_df["gauge_id"] = gauge_df["gauge_id"].astype(str).str.zfill(8)
basin_name_map = gauge_df.set_index("gauge_id")["gauge_name"].to_dict()

# Output PDF
heatmap_pdf = PDF_DIR / "TE_heatmap_all671_PET_to_Q_daymet_suffix05_bins5.pdf"

with PdfPages(str(heatmap_pdf)) as pdf:

    gauge_ids = sorted(te_df["gauge_id"].unique())

    for i, gauge_id in enumerate(gauge_ids, start=1):

        basin_df = te_df[te_df["gauge_id"] == gauge_id].copy()

        heat_df = basin_df.pivot_table(
            index="Source",
            columns="Lag",
            values="TE_nats",
            aggfunc="first"
        )

        basin_name = basin_name_map.get(gauge_id, "Unknown Basin")

        fig, ax = plt.subplots(figsize=(12, 2.2))

        sns.heatmap(
            heat_df,
            cmap="viridis",
            annot=True,
            fmt=".5f",
            annot_kws={"size": 5},
            linewidths=0.5,
            linecolor="white",
            cbar=True,
            cbar_kws={"label": "TE (nats)"},
            ax=ax
        )

        ax.set_title(f"Gauge ID: {gauge_id} | {basin_name} | PET(t-τ) → Q(t)", fontsize=10)
        ax.set_xlabel("Lag (days)")
        ax.set_ylabel("")
        ax.tick_params(axis="x", labelsize=8)
        ax.tick_params(axis="y", labelsize=8)

        plt.tight_layout()
        pdf.savefig(fig, dpi=100, bbox_inches="tight")
        plt.close(fig)
        gc.collect()

        if i % 50 == 0:
            print("Saved heatmaps:", i, "/", len(gauge_ids))

print("Saved PDF:", heatmap_pdf)

Saved heatmaps: 50 / 671
Saved heatmaps: 100 / 671
Saved heatmaps: 150 / 671
Saved heatmaps: 200 / 671
Saved heatmaps: 250 / 671
Saved heatmaps: 300 / 671
Saved heatmaps: 350 / 671
Saved heatmaps: 400 / 671
Saved heatmaps: 450 / 671
Saved heatmaps: 500 / 671
Saved heatmaps: 550 / 671
Saved heatmaps: 600 / 671
Saved heatmaps: 650 / 671
Saved PDF: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Transfer Entropy PET_to Q\outputs\pdf\TE_heatmap_all671_PET_to_Q_daymet_suffix05_bins5.pdf


## Define shuffle-test function for transfer entropy from PET(t-τ) to Q(t)

In [19]:
SFL = 200
MIN_POINTS = 50

def shuffle_test_te_full(x, y, bins, lag, sfl=200, rng=None, base=np.e):
    """
    Shuffle-test for transfer entropy T(X -> Y).

    Returns:
        te_obs  : observed TE
        thr95   : 95th percentile of shuffled TE
        pval    : proportion of shuffled TE >= observed TE
        sig     : 1 if observed TE > thr95 else 0
    """

    if rng is None:
        rng = np.random.default_rng(42)

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) <= lag or len(y) <= lag or len(x) < MIN_POINTS:
        return np.nan, np.nan, np.nan, 0

    te_obs = transfer_entropy(x, y, bins=bins, base=base, lag=lag)

    te_shuffled = []

    for _ in range(sfl):
        x_shuf = rng.permutation(x)
        te_s = transfer_entropy(x_shuf, y, bins=bins, base=base, lag=lag)
        te_shuffled.append(te_s)

    te_shuffled = np.array(te_shuffled, dtype=float)
    te_shuffled = te_shuffled[~np.isnan(te_shuffled)]

    if len(te_shuffled) == 0:
        return te_obs, np.nan, np.nan, 0

    thr95 = np.percentile(te_shuffled, 95)
    pval = np.mean(te_shuffled >= te_obs)
    sig = 1 if te_obs > thr95 else 0

    return te_obs, thr95, pval, sig

## Settings for PET to Q transfer entropy shuffle test

In [20]:
shuffle_csv = SUMMARY_DIR / "TE_shuffle_all671_tau1_30_PET_to_Q_daymet_suffix05_bins5.csv"
shuffle_log_csv = LOG_DIR / "TE_shuffle_all671_PET_to_Q_daymet_suffix05_log.csv"

shuffle_results = []
shuffle_log_rows = []

rng = np.random.default_rng(42)

## Run shuffle significance test for PET(t-τ) to Q(t) for all 671 basins

In [21]:
for csv_path in sorted(MERGED_DIR.glob("*_PET_Q_daymet_suffix05.csv")):

    gauge_id = csv_path.name.split("_")[0]

    try:
        df = pd.read_csv(csv_path, parse_dates=["Date"]).sort_values("Date")

        # numeric
        for col in ["PET", "Q"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        # keep original alignment; only mark invalid values
        df.loc[df["PET"] < 0, "PET"] = np.nan
        df.loc[df["Q"] < 0, "Q"] = np.nan

        # optional date filter
        if USE_CUSTOM_DATES:
            df = df[(df["Date"] >= start_date) & (df["Date"] <= end_date)].copy()

        if len(df) == 0:
            shuffle_log_rows.append([gauge_id, "NO DATA", 0, "empty after date filter"])
            continue

        x_all = df["PET"].values
        y_all = df["Q"].values

        for lag in LAGS:
            if len(x_all) <= lag or len(y_all) <= lag or len(x_all) < MIN_POINTS:
                te_obs, thr95, pval, sig = np.nan, np.nan, np.nan, 0
            else:
                te_obs, thr95, pval, sig = shuffle_test_te_full(
                    x_all,
                    y_all,
                    bins=NBINS,
                    lag=lag,
                    sfl=SFL,
                    rng=rng,
                    base=np.e
                )

            shuffle_results.append([
                gauge_id, "PET", "Q", NBINS, lag, te_obs, thr95, pval, sig
            ])

        shuffle_log_rows.append([gauge_id, "OK", len(df), ""])

        if len(shuffle_log_rows) % 50 == 0:
            print("Done basins:", len(shuffle_log_rows), "/ 671")

    except Exception as e:
        shuffle_log_rows.append([gauge_id, "FAIL", np.nan, str(e)])

shuffle_df = pd.DataFrame(
    shuffle_results,
    columns=["gauge_id", "Source", "Target", "Bins", "Lag", "TE_obs", "thr95", "p_value", "sig"]
)
shuffle_df.to_csv(shuffle_csv, index=False)

shuffle_log_df = pd.DataFrame(
    shuffle_log_rows,
    columns=["gauge_id", "status", "n_points_window", "info"]
)
shuffle_log_df.to_csv(shuffle_log_csv, index=False)

print("Saved shuffle CSV:", shuffle_csv.name)
print("Saved shuffle log CSV:", shuffle_log_csv.name)
print(shuffle_log_df["status"].value_counts())

display(shuffle_df.head())

Done basins: 50 / 671
Done basins: 100 / 671
Done basins: 150 / 671
Done basins: 200 / 671
Done basins: 250 / 671
Done basins: 300 / 671
Done basins: 350 / 671
Done basins: 400 / 671
Done basins: 450 / 671
Done basins: 500 / 671
Done basins: 550 / 671
Done basins: 600 / 671
Done basins: 650 / 671
Saved shuffle CSV: TE_shuffle_all671_tau1_30_PET_to_Q_daymet_suffix05_bins5.csv
Saved shuffle log CSV: TE_shuffle_all671_PET_to_Q_daymet_suffix05_log.csv
status
OK    671
Name: count, dtype: int64


,gauge_id,Source,Target,Bins,Lag,TE_obs,thr95,p_value,sig
0,01013500,PET,Q,5,1,0.002974,0.001648,0.0,1
1,01013500,PET,Q,5,2,0.003436,0.001694,0.0,1
2,01013500,PET,Q,5,3,0.003339,0.001600,0.0,1
3,01013500,PET,Q,5,4,0.003262,0.001595,0.0,1
4,01013500,PET,Q,5,5,0.003363,0.001743,0.0,1


## Save comparison PDF of original TE and shuffle-significant TE heatmaps for all basins

In [22]:
# Read original TE results
te_df = pd.read_csv(out_csv)
te_df["gauge_id"] = te_df["gauge_id"].astype(str).str.zfill(8)

# Read shuffle-test results
shuffle_df = pd.read_csv(shuffle_csv)
shuffle_df["gauge_id"] = shuffle_df["gauge_id"].astype(str).str.zfill(8)

# Keep only significant TE
shuffle_df["TE_sig"] = np.where(shuffle_df["sig"] == 1, shuffle_df["TE_obs"], np.nan)

# Basin name map
gauge_df["gauge_id"] = gauge_df["gauge_id"].astype(str).str.zfill(8)
basin_name_map = gauge_df.set_index("gauge_id")["gauge_name"].to_dict()

# Output PDF
compare_pdf = PDF_DIR / "TE_vs_ShuffleSig_heatmap_all671_PET_to_Q_daymet_suffix05_bins5.pdf"

with PdfPages(str(compare_pdf)) as pdf:

    gauge_ids = sorted(te_df["gauge_id"].unique())

    for i, gauge_id in enumerate(gauge_ids, start=1):

        # Basin data
        basin_te = te_df[te_df["gauge_id"] == gauge_id].copy()
        basin_shuf = shuffle_df[shuffle_df["gauge_id"] == gauge_id].copy()

        heat_te = basin_te.pivot_table(
            index="Source",
            columns="Lag",
            values="TE_nats",
            aggfunc="first"
        )

        heat_sig = basin_shuf.pivot_table(
            index="Source",
            columns="Lag",
            values="TE_sig",
            aggfunc="first"
        )

        basin_name = basin_name_map.get(gauge_id, "Unknown Basin")

        # Same color scale for both panels of this basin only
        vals1 = heat_te.to_numpy().flatten()
        vals2 = heat_sig.to_numpy().flatten()
        vals_all = np.concatenate([vals1, vals2])
        vals_all = vals_all[~np.isnan(vals_all)]

        if len(vals_all) > 0:
            vmin = vals_all.min()
            vmax = vals_all.max()
        else:
            vmin, vmax = 0, 1

        # avoid zero color range
        if np.isclose(vmin, vmax):
            vmax = vmin + 1e-12

        # Plot
        fig, axes = plt.subplots(1, 2, figsize=(18, 3.2))

        sns.heatmap(
            heat_te,
            cmap="viridis",
            annot=True,
            fmt=".4f",
            annot_kws={"size": 4},
            linewidths=0.5,
            linecolor="white",
            cbar=True,
            cbar_kws={"label": "TE (nats)"},
            vmin=vmin,
            vmax=vmax,
            ax=axes[0]
        )

        sns.heatmap(
            heat_sig,
            cmap="viridis",
            annot=True,
            fmt=".4f",
            annot_kws={"size": 4},
            linewidths=0.5,
            linecolor="white",
            cbar=True,
            cbar_kws={"label": "TE (nats)"},
            vmin=vmin,
            vmax=vmax,
            ax=axes[1]
        )

        axes[0].set_title("Original TE", fontsize=10)
        axes[1].set_title("Shuffle-significant TE", fontsize=10)

        for ax in axes:
            ax.set_xlabel("Lag (days)")
            ax.set_ylabel("")
            ax.tick_params(axis="x", labelsize=8)
            ax.tick_params(axis="y", labelsize=8)

        fig.suptitle(
            f"Gauge ID: {gauge_id} | {basin_name} | PET(t-τ) → Q(t)",
            fontsize=11,
            y=1.03
        )

        plt.tight_layout()
        pdf.savefig(fig, dpi=100, bbox_inches="tight")
        plt.close(fig)
        gc.collect()

        if i % 50 == 0:
            print("Saved comparison pages:", i, "/", len(gauge_ids))

print("Saved comparison PDF:", compare_pdf)

Saved comparison pages: 50 / 671


c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set(xlim=(0, self.data.shape[1]), ylim=(0, self.data.shape[0]))
c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set(xlim=(0, self.data.shape[1]), ylim=(0, self.data.shape[0]))
c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set(xlim=(0, self.data.shape[1]), ylim=(0, self.data.shape[0]))
c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high

Saved comparison pages: 100 / 671


c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set(xlim=(0, self.data.shape[1]), ylim=(0, self.data.shape[0]))
c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set(xlim=(0, self.data.shape[1]), ylim=(0, self.data.shape[0]))
c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set(xlim=(0, self.data.shape[1]), ylim=(0, self.data.shape[0]))
c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high

Saved comparison pages: 150 / 671


...

Saved comparison pages: 600 / 671


c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set(xlim=(0, self.data.shape[1]), ylim=(0, self.data.shape[0]))
c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set(xlim=(0, self.data.shape[1]), ylim=(0, self.data.shape[0]))


Saved comparison pages: 650 / 671


c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set(xlim=(0, self.data.shape[1]), ylim=(0, self.data.shape[0]))
c:\Users\ppaudel2\AppData\Local\anaconda3\envs\exercises\Lib\site-packages\seaborn\matrix.py:309: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.set(xlim=(0, self.data.shape[1]), ylim=(0, self.data.shape[0]))


Saved comparison PDF: C:\Users\ppaudel2\OneDrive - The University of Alabama\Desktop\Python\Research -CAMELS\Transfer Entropy PET_to Q\outputs\pdf\TE_vs_ShuffleSig_heatmap_all671_PET_to_Q_daymet_suffix05_bins5.pdf
